# Steel plant IIoT data generator

Generates interconnected synthetic data for a steel plant thermo-hydraulics predictive maintenance pipeline.

**Tables:** assets (once), sensor_readings, pid_control, alarm_events, work_orders (every run)

All logically connected: PID responds to readings, alarms from SPC with 5-min debounce, work orders aligned to anomaly trends with technician response delays.

In [ ]:
import json, random, math
from datetime import datetime, timedelta

VOLUME_BASE = "/Volumes/catalog1/iot_hello_world/raw_landing"
NUM_ASSETS = 15
RUN_DURATION_MINUTES = 60
SUBSECOND_WINDOW_SEC = 60  # first 60s get sub-second, rest is 1/sec

now = datetime.utcnow()
file_ts = now.strftime("%Y%m%d_%H%M%S")
run_start = now - timedelta(minutes=RUN_DURATION_MINUTES)
print(f"Generating {RUN_DURATION_MINUTES}min of data ending at {now.isoformat()}")

## 1. Assets (generate once)

In [ ]:
assets_dir = f"{VOLUME_BASE}/assets"
try:
    existing = dbutils.fs.ls(assets_dir)
    has_assets = len(existing) > 0
except:
    has_assets = False

cfgs = [
    ("pump","coolant_pump","hot_strip_mill","Siemens",250,280,12,150),
    ("pump","hydraulic_pump","rolling_mill","ABB",180,200,18,120),
    ("pump","feed_pump","blast_furnace","Emerson",90,320,10,180),
    ("valve","control_valve","reheat_furnace","Honeywell",45,350,8,200),
    ("valve","throttle_valve","caster","Yokogawa",60,260,14,130),
    ("heat_exchanger","shell_tube_hx","utilities","Danfoss",400,180,6,250),
    ("heat_exchanger","plate_hx","hot_strip_mill","Siemens",350,220,9,200),
    ("boiler","reheat_boiler","reheat_furnace","ABB",500,400,20,300),
    ("motor","rolling_mill_motor","rolling_mill","Siemens",300,150,5,80),
    ("compressor","air_compressor","utilities","Emerson",200,120,16,100),
    ("pump","coolant_pump","caster","Honeywell",220,270,11,160),
    ("valve","safety_valve","blast_furnace","Yokogawa",35,340,22,140),
    ("motor","conveyor_motor","hot_strip_mill","ABB",150,130,4,60),
    ("compressor","gas_compressor","blast_furnace","Danfoss",280,160,18,110),
    ("boiler","blast_furnace_boiler","blast_furnace","Siemens",480,420,24,280),
]

random.seed(42)
assets = []
for i,(at,sub,zone,mfr,cap,tsp,psp,fsp) in enumerate(cfgs,1):
    install = datetime(2018,1,1)+timedelta(days=random.randint(0,2000))
    assets.append({"asset_id":f"ASSET-{i:04d}","asset_name":f"{zone}_{sub}_{i}",
        "asset_type":at,"asset_subtype":sub,"zone":zone,"manufacturer":mfr,
        "install_date":install.strftime("%Y-%m-%d"),"rated_capacity_kw":cap,
        "temp_setpoint_c":tsp,"pressure_setpoint_bar":psp,"flow_setpoint_lpm":fsp})

if not has_assets:
    dbutils.fs.put(f"{assets_dir}/assets.json",
        "\n".join(json.dumps(a) for a in assets), overwrite=True)
    print(f"Created {len(assets)} assets")
else:
    print("Assets exist, skipping")

## 2. Plan incidents

In [ ]:
# R1: 3+sig spike, R2: 3+sig multi-spike cluster
# R3: 2+sig shift (long trend), R4: 1+sig bias (even longer)
incident_types = [
    {"type":"spike","sigma":3.5,"dur":(1,5),"pri":"critical"},
    {"type":"cluster","sigma":3.2,"dur":(5,15),"pri":"high"},
    {"type":"shift","sigma":2.3,"dur":(20,40),"pri":"medium"},
    {"type":"bias","sigma":1.3,"dur":(25,45),"pri":"low"},
]
n_inc = random.randint(2,4)
inc_idxs = random.sample(range(NUM_ASSETS), n_inc)
incidents = []
for aidx in inc_idxs:
    it = random.choice(incident_types)
    start = random.randint(5, RUN_DURATION_MINUTES-20)
    dur = random.randint(*it["dur"])
    resp_delay = random.randint(20,40)
    fix_dur = random.randint(15,60)
    incidents.append({"aidx":aidx,"type":it["type"],"sigma":it["sigma"],
        "start":start,"dur":dur,"resp_delay":resp_delay,
        "fix_dur":fix_dur,"pri":it["pri"]})
inc_map = {i["aidx"]:i for i in incidents}
for i in incidents:
    a=assets[i['aidx']]
    print(f"{a['asset_id']} ({a['asset_subtype']}): {i['type']} {i['sigma']}sig @ min{i['start']}, dur{i['dur']}min, tech@{i['start']+i['resp_delay']}min, fixed@{i['start']+i['resp_delay']+i['fix_dur']}min")

## 3. Generate readings + PID + alarms

In [ ]:
all_readings = []
all_pid = []
all_alarms = []
alm_n = 0
Kp,Ki,Kd = 0.8,0.1,0.05
DEBOUNCE = 300  # 5min debounce between alarms per asset+rule
total_sec = RUN_DURATION_MINUTES * 60

for aidx,asset in enumerate(assets):
    sp = asset["temp_setpoint_c"]
    nstd = sp * 0.02
    hist = []
    integ = 0.0
    prev_e = 0.0
    inc = inc_map.get(aidx)
    last_alm = {}  # debounce tracker

    for sec in range(total_sec):
        cur_min = sec / 60.0
        nr = random.randint(5,10) if sec < SUBSECOND_WINDOW_SEC else 1
        for sub in range(nr):
            frac = sub/max(nr,1)
            ts = run_start + timedelta(seconds=sec, milliseconds=int(frac*1000))
            temp = sp + random.gauss(0, nstd)

            # Incident distortion
            if inc:
                i_end_fix = inc["start"] + inc["resp_delay"] + inc["fix_dur"]
                if inc["start"] <= cur_min < i_end_fix:
                    prog = min(1.0, (cur_min-inc["start"])/max(inc["dur"],1))
                    tech_at = inc["start"] + inc["resp_delay"]
                    if cur_min >= tech_at:
                        damp = max(0, 1.0 - (cur_min-tech_at)/max(inc["fix_dur"],1))
                    else:
                        damp = 1.0
                    if inc["type"]=="spike":
                        if random.random()<0.05: temp = sp + inc["sigma"]*nstd*random.choice([1,-1])*damp
                    elif inc["type"]=="cluster":
                        if random.random()<0.3: temp = sp + inc["sigma"]*nstd*random.choice([1,-1])*damp
                    elif inc["type"]=="shift":
                        temp = sp + inc["sigma"]*nstd*damp + random.gauss(0, nstd*0.3)
                    elif inc["type"]=="bias":
                        temp = sp + inc["sigma"]*nstd*prog*damp + random.gauss(0, nstd*0.4)

            temp = round(temp, 2)
            hist.append(temp)
            if len(hist)>500: hist = hist[-500:]

            # SPC
            r1=r2=r3=r4=anom=False
            if len(hist)>=20:
                hm = sum(hist)/len(hist)
                hs = max((sum((v-hm)**2 for v in hist)/len(hist))**0.5, 0.001)
                dv = abs(temp-hm)/hs
                r1 = dv > 3.0
                anom = r1
                if len(hist)>=5:
                    l5=hist[-5:]
                    r2 = sum(1 for v in l5 if v>hm+hs)>=4 or sum(1 for v in l5 if v<hm-hs)>=4
                if len(hist)>=9:
                    l9=hist[-9:]
                    r3 = all(v>hm for v in l9) or all(v<hm for v in l9)
                if len(hist)>=14:
                    l14=hist[-14:]
                    ic=sum(1 for j in range(13) if l14[j+1]>l14[j])
                    r4 = ic>=12 or ic<=2

            te = (temp-sp)/max(nstd,0.1)
            pr = round(asset["pressure_setpoint_bar"]+random.gauss(0,0.5)+te*0.3, 3)
            fl = round(max(0, asset["flow_setpoint_lpm"]+random.gauss(0,5)-te*2), 2)
            vb = round(max(0, 2.0+random.gauss(0,0.3)+abs(te)*0.15), 3)
            pw = round(max(0, asset["rated_capacity_kw"]*0.7+random.gauss(0,10)+te*5), 2)
            tss = ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

            all_readings.append({"asset_id":asset["asset_id"],"timestamp":tss,
                "temperature_c":temp,"pressure_bar":pr,"flow_rate_lpm":fl,
                "vibration_mm_s":vb,"power_draw_kw":pw,"anomaly_3sig":anom,
                "spc_r1_spike":r1,"spc_r2_cluster":r2,"spc_r3_shift":r3,"spc_r4_bias":r4})

            # PID
            err = sp - temp
            integ += err*0.1
            integ = max(-50, min(50, integ))
            deriv = (err-prev_e)/0.1
            po = max(-100, min(100, Kp*err + Ki*integ + Kd*deriv))
            all_pid.append({"asset_id":asset["asset_id"],"timestamp":tss,
                "setpoint_temp_c":sp,"process_temp_c":temp,
                "error_c":round(err,2),"pid_output":round(po,2),
                "p_term":round(Kp*err,2),"i_term":round(Ki*integ,2),"d_term":round(Kd*deriv,2),
                "valve_position_pct":round(max(0,min(100,50+po*0.5)),1),
                "actuator_power_pct":round(max(0,min(100,abs(po))),1),
                "cooling_flow_cmd_lpm":round(max(0,asset["flow_setpoint_lpm"]+po*1.5),1)})
            prev_e = err

            # Alarms with debounce
            for rn,rf,sv in [("r1_spike",r1,"high"),("r2_cluster",r2,"high"),
                             ("r3_shift",r3,"medium"),("r4_bias",r4,"low")]:
                if rf:
                    dk = f"{asset['asset_id']}_{rn}"
                    lt = last_alm.get(dk, -9999)
                    if sec - lt >= DEBOUNCE:
                        fa = random.random()<0.2
                        disp = (not fa) and random.random()<0.75
                        all_alarms.append({"alarm_id":f"ALM-{file_ts}-{alm_n:05d}",
                            "asset_id":asset["asset_id"],"timestamp":tss,
                            "trigger_type":f"spc_{rn}","severity":sv,
                            "process_value_c":temp,"dispatched":disp,
                            "resolution_minutes":random.randint(20,45) if disp else None,
                            "false_alarm":fa})
                        alm_n += 1
                        last_alm[dk] = sec

print(f"Readings: {len(all_readings)}, PID: {len(all_pid)}, Alarms: {len(all_alarms)}")

## 4. Work orders

In [ ]:
wo = []
for wi,inc in enumerate(incidents):
    a = assets[inc["aidx"]]
    cr = run_start + timedelta(minutes=inc["start"]+inc["resp_delay"])
    co = cr + timedelta(minutes=inc["fix_dur"])
    if inc["sigma"]>=3.0: pri="critical" if inc["sigma"]>=3.5 else "high"; ot="corrective"
    elif inc["sigma"]>=2.0: pri,ot="medium","corrective"
    else: pri,ot="low","inspection"
    wo.append({"work_order_id":f"WO-{file_ts}-{wi:05d}","asset_id":a["asset_id"],
        "order_type":ot,"priority":pri,"created_at":cr.strftime("%Y-%m-%dT%H:%M:%S"),
        "scheduled_date":cr.strftime("%Y-%m-%d"),"completed_at":co.strftime("%Y-%m-%dT%H:%M:%S"),
        "status":"completed","description":f"{inc['type']} event on {a['asset_subtype']} in {a['zone']}",
        "resolution_minutes":inc["fix_dur"]})

for i in range(random.randint(3,6)):
    a = random.choice(assets)
    cr = run_start - timedelta(days=random.randint(1,14))
    sc = cr + timedelta(days=random.randint(1,7))
    wo.append({"work_order_id":f"WO-{file_ts}-R{i:03d}","asset_id":a["asset_id"],
        "order_type":random.choice(["preventive","inspection"]),
        "priority":random.choice(["low","medium"]),
        "created_at":cr.strftime("%Y-%m-%dT%H:%M:%S"),
        "scheduled_date":sc.strftime("%Y-%m-%d"),
        "completed_at":sc.strftime("%Y-%m-%dT%H:%M:%S"),
        "status":random.choice(["completed","in_progress","open"]),
        "description":f"Routine maintenance on {a['asset_subtype']}",
        "resolution_minutes":random.randint(30,120)})
print(f"Work orders: {len(wo)}")

## 5. Write to Volume

In [ ]:
for name,data in [("sensor_readings",all_readings),("pid_control",all_pid),
                   ("alarm_events",all_alarms),("work_orders",wo)]:
    if data:
        ndjson = "\n".join(json.dumps(r) for r in data)
        path = f"{VOLUME_BASE}/{name}/{file_ts}.json"
        dbutils.fs.put(path, ndjson, overwrite=True)
        print(f"{name}: {len(data)} records -> {path}")
print("Done!")